# Mansoni Orchestrator — Bootstrap & Bridge

Этот ноутбук:
1. Создаёт структуру директорий для bridge-модулей
2. Записывает все необходимые Python-файлы
3. Тестирует импорт оркестратора
4. Запускает тестовую задачу

In [ ]:
import os
import sys

PROJECT_ROOT = r'c:\Users\manso\Desktop\разработка\your-ai-companion-main'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

# Создаём структуру директорий
dirs = [
    'ai_engine/bridge',
    'ai_engine/bridge/tools',
    'ai_engine/bridge/agents',
    'ai_engine/bridge/reports',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    init = os.path.join(d, '__init__.py')
    if not os.path.exists(init):
        with open(init, 'w') as f:
            f.write('')

print('Директории созданы:')
for d in dirs:
    print(f'  ✓ {d}')

In [ ]:
# ── tool_implementations.py ─────────────────────────────────────
# Реальные инструменты для агентов: файлы, терминал, grep, tsc

TOOL_IMPL_CODE = r'''
#!/usr/bin/env python3
"""Реальные инструменты для агентов оркестратора.

Каждый инструмент — callable, который агент может вызвать.
Все операции — на реальной файловой системе проекта.
"""

import os
import re
import subprocess
import json
import shlex
from pathlib import Path
from typing import Optional


class ProjectTools:
    """Набор инструментов, привязанных к конкретному проекту."""

    def __init__(self, project_root: str) -> None:
        self.root = Path(project_root).resolve()
        if not self.root.exists():
            raise FileNotFoundError(f"Project root not found: {self.root}")

    def _safe_path(self, rel: str) -> Path:
        """Проверка path traversal."""
        full = (self.root / rel).resolve()
        if not str(full).startswith(str(self.root)):
            raise ValueError(f"Path traversal attempt: {rel}")
        return full

    # ── File Operations ──────────────────────────────────────────

    def read_file(self, path: str, start_line: int = 0, end_line: int = 0) -> str:
        """Прочитать файл (целиком или диапазон строк)."""
        fp = self._safe_path(path)
        if not fp.exists():
            return f"ERROR: File not found: {path}"
        text = fp.read_text(encoding="utf-8", errors="replace")
        if start_line > 0 or end_line > 0:
            lines = text.splitlines(keepends=True)
            s = max(start_line - 1, 0)
            e = end_line if end_line > 0 else len(lines)
            return "".join(lines[s:e])
        return text

    def write_file(self, path: str, content: str) -> str:
        """Записать/создать файл."""
        fp = self._safe_path(path)
        fp.parent.mkdir(parents=True, exist_ok=True)
        fp.write_text(content, encoding="utf-8")
        return f"OK: Written {len(content)} chars to {path}"

    def replace_in_file(self, path: str, old: str, new: str) -> str:
        """Заменить строку в файле."""
        fp = self._safe_path(path)
        if not fp.exists():
            return f"ERROR: File not found: {path}"
        text = fp.read_text(encoding="utf-8")
        if old not in text:
            return f"ERROR: Pattern not found in {path}"
        count = text.count(old)
        text = text.replace(old, new, 1)
        fp.write_text(text, encoding="utf-8")
        return f"OK: Replaced 1/{count} occurrence(s) in {path}"

    def list_files(self, directory: str = ".", pattern: str = "*") -> str:
        """Список файлов в директории."""
        dp = self._safe_path(directory)
        if not dp.is_dir():
            return f"ERROR: Not a directory: {directory}"
        files = sorted(str(f.relative_to(self.root)) for f in dp.rglob(pattern) if f.is_file())
        return "\n".join(files[:500])  # лимит

    def file_line_count(self, path: str) -> str:
        """Количество строк в файле."""
        fp = self._safe_path(path)
        if not fp.exists():
            return f"ERROR: File not found: {path}"
        count = len(fp.read_text(encoding="utf-8", errors="replace").splitlines())
        return str(count)

    # ── Search ───────────────────────────────────────────────────

    def grep(self, pattern: str, include: str = "*.ts,*.tsx,*.py", max_results: int = 50) -> str:
        """Поиск по регулярному выражению в файлах проекта."""
        regex = re.compile(pattern, re.IGNORECASE)
        exts = set(include.split(","))
        results = []
        ignore_dirs = {"node_modules", ".git", "dist", "build", "__pycache__", ".venv", "reserve", "archive"}
        for root_dir, dirs, files in os.walk(self.root):
            dirs[:] = [d for d in dirs if d not in ignore_dirs]
            for fname in files:
                if not any(fname.endswith(ext.replace("*", "")) for ext in exts):
                    continue
                fpath = Path(root_dir) / fname
                try:
                    for i, line in enumerate(fpath.read_text(encoding="utf-8", errors="replace").splitlines(), 1):
                        if regex.search(line):
                            rel = str(fpath.relative_to(self.root)).replace("\\\\", "/")
                            results.append(f"{rel}:{i}: {line.strip()[:120]}")
                            if len(results) >= max_results:
                                return "\n".join(results)
                except (OSError, UnicodeDecodeError):
                    pass
        return "\n".join(results) if results else "No matches found"

    # ── Terminal ─────────────────────────────────────────────────

    def run_command(self, command: str, timeout: int = 120) -> str:
        """Выполнить команду в терминале (с ограничениями безопасности)."""
        # Блокируем опасные команды
        blocked = ["rm -rf /", "del /s /q c:\\\\", "format ", "DROP DATABASE", "--force", "sudo rm"]
        for b in blocked:
            if b.lower() in command.lower():
                return f"BLOCKED: Dangerous command detected: {b}"
        try:
            result = subprocess.run(
                command, shell=True, capture_output=True, text=True,
                timeout=timeout, cwd=str(self.root),
                env={**os.environ, "FORCE_COLOR": "0"},
            )
            output = result.stdout + result.stderr
            exit_code = result.returncode
            return f"Exit: {exit_code}\n{output[-3000:]}"  # лимит вывода
        except subprocess.TimeoutExpired:
            return f"TIMEOUT: Command exceeded {timeout}s"
        except Exception as e:
            return f"ERROR: {e}"

    def tsc_check(self) -> str:
        """Проверить TypeScript: npx tsc -p tsconfig.app.json --noEmit."""
        return self.run_command("npx tsc -p tsconfig.app.json --noEmit", timeout=180)

    def lint_check(self) -> str:
        """Запустить ESLint."""
        return self.run_command("npm run lint", timeout=120)

    def run_tests(self, pattern: str = "") -> str:
        """Запустить vitest."""
        cmd = "npx vitest run --reporter=verbose"
        if pattern:
            cmd += f" {pattern}"
        return self.run_command(cmd, timeout=300)

    # ── Memory ───────────────────────────────────────────────────

    def read_memory(self, scope: str = "repo") -> str:
        """Прочитать память проекта из /memories/{scope}/."""
        mem_dir = self._safe_path(f"memories/{scope}" if scope != "session" else f"memories/session")
        # Fallback: check .memories/ or memories/ relative paths
        possible = [self.root / "memories" / scope, self.root / ".memories" / scope]
        for p in possible:
            if p.is_dir():
                files = sorted(p.glob("*.md"))
                result = []
                for f in files:
                    result.append(f"### {f.name}\n{f.read_text(encoding=\"utf-8\", errors=\"replace\")[:2000]}")
                return "\n\n".join(result) if result else "Empty memory"
        return f"Memory directory not found for scope: {scope}"

    def write_memory(self, filename: str, content: str, scope: str = "repo") -> str:
        """Записать в память проекта."""
        mem_dir = self.root / "memories" / scope
        mem_dir.mkdir(parents=True, exist_ok=True)
        fp = mem_dir / filename
        fp.write_text(content, encoding="utf-8")
        return f"OK: Memory written to {scope}/{filename}"
'''

with open('ai_engine/bridge/tool_implementations.py', 'w', encoding='utf-8') as f:
    f.write(TOOL_IMPL_CODE.strip())
print('✓ tool_implementations.py записан')

In [ ]:
# ── llm_provider.py ─────────────────────────────────────────────
# Подключение к Claude/OpenAI API для reasoning агентов

LLM_PROVIDER_CODE = r'''
#!/usr/bin/env python3
"""LLM Provider — подключение к Claude или OpenAI.

Поддерживает:
    - Anthropic Claude (claude-sonnet-4-20250514 / claude-opus-4-20250514)
    - OpenAI GPT-4o
    - Fallback: простой эвристический режим
"""

import json
import os
import logging
from typing import Optional, Callable

logger = logging.getLogger(__name__)


def create_llm_callable(
    provider: str = "auto",
    model: Optional[str] = None,
    api_key: Optional[str] = None,
    max_tokens: int = 4096,
    temperature: float = 0.3,
) -> Callable[[str], str]:
    """Создать callable для LLM.
    
    Args:
        provider: "anthropic", "openai", "auto" (определяет по ключу)
        model: ID модели (или auto-detect)
        api_key: API ключ (или из env)
    
    Returns:
        Callable[[str], str] — функция prompt -> response
    """
    # Определяем провайдер
    key = api_key or os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY", "")
    
    if provider == "auto":
        if os.environ.get("ANTHROPIC_API_KEY") or (key and key.startswith("sk-ant-")):
            provider = "anthropic"
        elif os.environ.get("OPENAI_API_KEY") or (key and key.startswith("sk-")):
            provider = "openai"
        else:
            provider = "heuristic"
            logger.warning("Нет API ключа — используется эвристический режим")

    if provider == "anthropic":
        return _create_anthropic_callable(key, model or "claude-sonnet-4-20250514", max_tokens, temperature)
    elif provider == "openai":
        return _create_openai_callable(key, model or "gpt-4o", max_tokens, temperature)
    else:
        return _heuristic_fallback


def _create_anthropic_callable(
    api_key: str, model: str, max_tokens: int, temperature: float
) -> Callable[[str], str]:
    """Claude API через httpx (без SDK — минимум зависимостей)."""
    try:
        import httpx
    except ImportError:
        import urllib.request
        import ssl
        
        def call_urllib(prompt: str) -> str:
            data = json.dumps({
                "model": model,
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": [{"role": "user", "content": prompt}],
            }).encode("utf-8")
            req = urllib.request.Request(
                "https://api.anthropic.com/v1/messages",
                data=data,
                headers={
                    "Content-Type": "application/json",
                    "x-api-key": api_key,
                    "anthropic-version": "2023-06-01",
                },
            )
            ctx = ssl.create_default_context()
            with urllib.request.urlopen(req, timeout=120, context=ctx) as resp:
                body = json.loads(resp.read().decode("utf-8"))
                return body["content"][0]["text"]
        return call_urllib

    def call_httpx(prompt: str) -> str:
        resp = httpx.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "Content-Type": "application/json",
                "x-api-key": api_key,
                "anthropic-version": "2023-06-01",
            },
            json={
                "model": model,
                "max_tokens": max_tokens,
                "temperature": temperature,
                "messages": [{"role": "user", "content": prompt}],
            },
            timeout=120,
        )
        resp.raise_for_status()
        return resp.json()["content"][0]["text"]
    return call_httpx


def _create_openai_callable(
    api_key: str, model: str, max_tokens: int, temperature: float
) -> Callable[[str], str]:
    """OpenAI API через urllib (stdlib)."""
    import urllib.request
    import ssl
    
    def call(prompt: str) -> str:
        data = json.dumps({
            "model": model,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "messages": [{"role": "user", "content": prompt}],
        }).encode("utf-8")
        req = urllib.request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=data,
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {api_key}",
            },
        )
        ctx = ssl.create_default_context()
        with urllib.request.urlopen(req, timeout=120, context=ctx) as resp:
            body = json.loads(resp.read().decode("utf-8"))
            return body["choices"][0]["message"]["content"]
    return call


def _heuristic_fallback(prompt: str) -> str:
    """Эвристический фолбек без API."""
    lower = prompt.lower()
    if "разбей" in lower or "декомпози" in lower:
        return json.dumps({"subtasks": ["Анализ", "Реализация", "Тестирование", "Ревью"]}, ensure_ascii=False)
    if "оцени" in lower or "quality" in lower:
        return json.dumps({"score": 0.7, "issues": ["Требуется ручная проверка"]}, ensure_ascii=False)
    return "Эвристический режим: требуется API ключ для полноценной работы"
'''

with open('ai_engine/bridge/llm_provider.py', 'w', encoding='utf-8') as f:
    f.write(LLM_PROVIDER_CODE.strip())
print('✓ llm_provider.py записан')

In [ ]:
# ── agent_factory.py ─────────────────────────────────────────────
# Создание специализированных агентов с реальными инструментами

AGENT_FACTORY_CODE = r'''
#!/usr/bin/env python3
"""Agent Factory — создание агентов с реальными инструментами.

Каждый агент получает:
    - Специализированный system prompt (скилл)
    - Доступ к ProjectTools
    - LLM callable для reasoning
    - Собственную рабочую память
"""

import json
import logging
from typing import Any, Callable, Optional
from pathlib import Path

logger = logging.getLogger(__name__)


# ── System Prompts (скиллы) агентов ───────────────────────────────

AGENT_PROMPTS = {
    "reviewer": """Ты — Code Reviewer. Аудит по 8 направлениям:
1. TypeScript strict (tsc ошибки)
2. Безопасность (XSS, IDOR, injection)
3. Производительность (рендеры, N+1, bundle size)
4. Архитектура (размер файлов >400, дубли, мёртвый код)
5. Error handling (пустые catch, fake success)
6. UX-состояния (loading/empty/error/offline)
7. Supabase (RLS, .limit(), .single())
8. Humanization (AI-паттерны)

Формат ответа: JSON с полями: direction, severity (critical/high/medium/low), file, line, description, fix.
Confidence score 0-100 для каждого направления.""",

    "debugger": """Ты — Debugger. Протокол: REPRODUCE → ISOLATE → ROOT CAUSE → FIX → VERIFY.
Всегда указывай файл:строку. Проверяй гипотезы grep-ом. Не гадай — ищи evidence.
После фикса запускай tsc_check для верификации.""",

    "tester": """Ты — Test Engineer. Пишешь тесты на vitest:
- Unit: изолированная логика, моки Supabase
- Integration: цепочки хуков/компонентов
- Edge cases: пустой массив, null user, timeout, offline
Формат: describe/it/expect. Запускай run_tests для верификации.""",

    "doc_writer": """Ты — Documentation Specialist. Генерируешь:
- Architecture docs (C4 diagrams в текстовом виде)
- API docs (endpoint, params, response, errors)
- Module docs (назначение, exports, dependencies)
- Schema docs (таблицы, RLS, relations)
Формат: Markdown, в директорию docs/.""",

    "architect": """Ты — Software Architect. Проектируешь:
- Модели данных (PostgreSQL + Supabase)
- API контракты (Edge Functions)
- Компонентную архитектуру (React)
- Интеграционные цепочки (UI → API → DB → side effects)
Всегда ищи существующие аналоги перед созданием нового.""",

    "codesmith": """Ты — Senior Developer. Пишешь production-ready код:
- TypeScript strict
- Error handling на boundaries
- Proper loading/empty/error states
- No stubs, no fake success
- Humanized: разный размер функций, минимум комментариев, короткие имена
После каждого изменения: tsc_check.""",

    "security": """Ты — Security Auditor. Проверяешь:
- OWASP Top 10
- Supabase RLS на каждой таблице
- E2EE реализация (ключи, шифрование, MITM)
- Edge Functions (auth, CORS, input validation)
- XSS через dangerouslySetInnerHTML
- IDOR через отсутствие проверки ownership
Формат: CVSS score, severity, remediation.""",
}


class AgentRunner:
    """Запускает агента с инструментами и LLM."""

    def __init__(
        self,
        role: str,
        tools: Any,  # ProjectTools
        llm: Callable[[str], str],
        system_prompt: Optional[str] = None,
    ) -> None:
        self.role = role
        self.tools = tools
        self.llm = llm
        self.system_prompt = system_prompt or AGENT_PROMPTS.get(role, "")
        self.history: list[dict[str, str]] = []
        self.report: list[str] = []

    def think(self, task: str) -> str:
        """Один шаг reasoning: задать вопрос LLM."""
        context = f"""SYSTEM: {self.system_prompt}

ДОСТУПНЫЕ ИНСТРУМЕНТЫ:
- read_file(path, start_line?, end_line?) → содержимое файла
- write_file(path, content) → записать файл
- replace_in_file(path, old, new) → заменить строку
- grep(pattern, include?) → поиск по кодовой базе
- tsc_check() → проверка TypeScript
- lint_check() → запуск ESLint
- run_tests(pattern?) → запуск vitest
- run_command(command) → терминал
- list_files(directory, pattern?) → список файлов
- read_memory(scope?) → прочитать память проекта

ФОРМАТ ОТВЕТА (строго JSON):
{{
  "thought": "...",
  "action": "tool_name",
  "args": {{...}},
  "is_final": false
}}
или финальный:
{{
  "thought": "...",
  "result": "...",
  "is_final": true
}}

ИСТОРИЯ:
{json.dumps(self.history[-20:], ensure_ascii=False)}

ЗАДАЧА: {task}"""
        return self.llm(context)

    def execute_tool(self, action: str, args: dict) -> str:
        """Выполнить инструмент."""
        tool_map = {
            "read_file": self.tools.read_file,
            "write_file": self.tools.write_file,
            "replace_in_file": self.tools.replace_in_file,
            "grep": self.tools.grep,
            "tsc_check": self.tools.tsc_check,
            "lint_check": self.tools.lint_check,
            "run_tests": self.tools.run_tests,
            "run_command": self.tools.run_command,
            "list_files": self.tools.list_files,
            "file_line_count": self.tools.file_line_count,
            "read_memory": self.tools.read_memory,
            "write_memory": self.tools.write_memory,
        }
        fn = tool_map.get(action)
        if not fn:
            return f"ERROR: Unknown tool: {action}"
        try:
            result = fn(**args)
            return str(result)[:5000]  # лимит вывода
        except Exception as e:
            return f"ERROR: {e}"

    def run(self, task: str, max_steps: int = 25) -> str:
        """Запустить агента на задаче (ReAct loop)."""
        logger.info(f"Agent [{self.role}] starting: {task[:80]}")
        for step in range(max_steps):
            try:
                raw = self.think(task)
                # Парсим JSON из ответа LLM
                response = self._parse_json(raw)
                if not response:
                    self.history.append({"step": str(step), "error": "Failed to parse LLM response", "raw": raw[:500]})
                    continue
                
                thought = response.get("thought", "")
                self.report.append(f"Step {step}: {thought}")
                
                if response.get("is_final"):
                    result = response.get("result", "")
                    self.history.append({"step": str(step), "thought": thought, "result": result})
                    logger.info(f"Agent [{self.role}] completed in {step+1} steps")
                    return result
                
                action = response.get("action", "")
                args = response.get("args", {})
                observation = self.execute_tool(action, args)
                self.history.append({
                    "step": str(step), "thought": thought,
                    "action": action, "args": str(args)[:200],
                    "observation": observation[:1000],
                })
                self.report.append(f"  → {action}({args}) → {observation[:200]}")
            except Exception as e:
                self.history.append({"step": str(step), "error": str(e)})
                self.report.append(f"  ⚠ Error: {e}")
        
        return f"Agent [{self.role}] reached max steps ({max_steps}). Partial report:\n" + "\n".join(self.report)

    @staticmethod
    def _parse_json(text: str) -> Optional[dict]:
        """Извлечь JSON из текста LLM."""
        # Ищем JSON блок
        import re
        patterns = [
            r"```json\s*\n(.*?)```",
            r"```\s*\n(.*?)```",
            r"(\{[\s\S]*\})",
        ]
        for pat in patterns:
            match = re.search(pat, text, re.DOTALL)
            if match:
                try:
                    return json.loads(match.group(1))
                except json.JSONDecodeError:
                    continue
        return None


def create_agent(role: str, tools: Any, llm: Callable[[str], str]) -> AgentRunner:
    """Фабрика агентов."""
    return AgentRunner(role=role, tools=tools, llm=llm)
'''

with open('ai_engine/bridge/agent_factory.py', 'w', encoding='utf-8') as f:
    f.write(AGENT_FACTORY_CODE.strip())
print('✓ agent_factory.py записан')

In [ ]:
# ── mansoni_runner.py ────────────────────────────────────────────
# Главная точка входа: CLI для запуска оркестратора

RUNNER_CODE = r'''
#!/usr/bin/env python3
"""Mansoni Runner — CLI для запуска мультиагентного оркестратора.

Использование:
    python -m ai_engine.bridge.mansoni_runner "задача текстом"
    python -m ai_engine.bridge.mansoni_runner --task-file tasks/audit.md
    python -m ai_engine.bridge.mansoni_runner --interactive

Пайплайн:
    1. Инициализация ProjectTools + LLM
    2. Создание агентов через AgentFactory
    3. Запуск OrchestratorCore
    4. Делегирование подзадач агентам
    5. Сбор отчётов
"""

import argparse
import json
import logging
import os
import sys
import time
from datetime import datetime
from pathlib import Path
from typing import Optional

# Добавляем корень проекта в sys.path
_this_dir = Path(__file__).resolve().parent
_project_root = _this_dir.parent.parent
sys.path.insert(0, str(_project_root))

from ai_engine.bridge.tool_implementations import ProjectTools
from ai_engine.bridge.llm_provider import create_llm_callable
from ai_engine.bridge.agent_factory import AgentRunner, create_agent, AGENT_PROMPTS

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
)
logger = logging.getLogger("mansoni")


class MansoniOrchestrator:
    """Координатор мультиагентного пайплайна."""

    def __init__(self, project_root: str, llm_provider: str = "auto") -> None:
        self.root = Path(project_root).resolve()
        self.tools = ProjectTools(str(self.root))
        self.llm = create_llm_callable(provider=llm_provider)
        self.agents: dict[str, AgentRunner] = {}
        self.reports_dir = self.root / "ai_engine" / "bridge" / "reports"
        self.reports_dir.mkdir(parents=True, exist_ok=True)
        
        # Регистрация агентов
        for role in AGENT_PROMPTS:
            self.agents[role] = create_agent(role, self.tools, self.llm)
        
        logger.info(f"Mansoni инициализирован. Агентов: {len(self.agents)}. Корень: {self.root}")

    def classify_task(self, task: str) -> dict:
        """Классифицировать задачу и выбрать пайплайн."""
        lower = task.lower()
        
        if any(w in lower for w in ["аудит", "ревью", "проверь", "audit", "review"]):
            return {
                "type": "audit",
                "pipeline": ["reviewer", "security", "doc_writer"],
                "description": "Комплексный аудит",
            }
        elif any(w in lower for w in ["баг", "ошибка", "fix", "bug", "сломал", "не работает"]):
            return {
                "type": "bug",
                "pipeline": ["debugger", "codesmith", "tester"],
                "description": "Диагностика и фикс",
            }
        elif any(w in lower for w in ["фича", "добавь", "создай", "реализуй", "feature", "implement"]):
            return {
                "type": "feature",
                "pipeline": ["architect", "codesmith", "tester", "reviewer"],
                "description": "Новая фича",
            }
        elif any(w in lower for w in ["рефакторинг", "упрости", "оптимизируй", "декомпозиция", "refactor"]):
            return {
                "type": "refactor",
                "pipeline": ["reviewer", "codesmith", "tester"],
                "description": "Рефакторинг",
            }
        elif any(w in lower for w in ["документ", "doc", "описа"]):
            return {
                "type": "docs",
                "pipeline": ["doc_writer"],
                "description": "Документация",
            }
        elif any(w in lower for w in ["тест", "test", "покрытие", "coverage"]):
            return {
                "type": "test",
                "pipeline": ["tester"],
                "description": "Тестирование",
            }
        else:
            return {
                "type": "general",
                "pipeline": ["reviewer"],
                "description": "Общая задача",
            }

    def run(self, task: str) -> str:
        """Запустить полный пайплайн."""
        start = time.time()
        classification = self.classify_task(task)
        
        header = f"""
╔══════════════════════════════════════════════════════════════╗
║  MANSONI | {classification["description"]:^48} ║
║  Тип: {classification["type"]:^52} ║
║  Агенты: {" → ".join(classification["pipeline"]):^49} ║
╚══════════════════════════════════════════════════════════════╝
"""
        print(header)
        logger.info(f"Задача: {task[:100]}")
        logger.info(f"Пайплайн: {' → '.join(classification['pipeline'])}")
        
        # Pre-flight: прочитать память
        memory = self.tools.read_memory("repo")
        logger.info(f"Память загружена: {len(memory)} символов")
        
        results = []
        accumulated_context = f"Задача: {task}\n\nПамять проекта (кратко):\n{memory[:3000]}"
        
        for i, role in enumerate(classification["pipeline"], 1):
            agent = self.agents.get(role)
            if not agent:
                logger.warning(f"Агент {role} не найден, пропускаю")
                continue
            
            print(f"\n{'─'*60}")
            print(f"  [{i}/{len(classification['pipeline'])}] Агент: {role}")
            print(f"{'─'*60}")
            
            # Передаём накопленный контекст
            agent_task = f"{accumulated_context}\n\nТвоя роль: {role}"
            result = agent.run(agent_task)
            results.append({"role": role, "result": result, "report": agent.report})
            
            # Добавляем результат в контекст для следующего агента
            accumulated_context += f"\n\nРезультат агента {role}:\n{result[:2000]}"
            
            print(f"  ✓ {role} завершён ({len(agent.report)} шагов)")
        
        elapsed = time.time() - start
        
        # Собираем финальный отчёт
        report = self._build_report(task, classification, results, elapsed)
        
        # Сохраняем отчёт
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        report_path = self.reports_dir / f"report_{classification['type']}_{ts}.md"
        report_path.write_text(report, encoding="utf-8")
        print(f"\n📄 Отчёт: {report_path.relative_to(self.root)}")
        print(f"⏱  Время: {elapsed:.1f}с")
        
        return report

    def _build_report(self, task: str, classification: dict, results: list, elapsed: float) -> str:
        """Сформировать полный отчёт."""
        ts = datetime.now().strftime("%Y-%m-%d %H:%M")
        sections = [f"# Отчёт Mansoni — {classification['description']}"]
        sections.append(f"""\n**Дата:** {ts}  \n**Задача:** {task}  \n**Тип:** {classification['type']}  \n**Пайплайн:** {' → '.join(classification['pipeline'])}  \n**Время:** {elapsed:.1f}с\n""")
        
        for r in results:
            sections.append(f"\n## Агент: {r['role']}\n")
            sections.append(r["result"])
            if r["report"]:
                sections.append(f"\n### Шаги выполнения\n")
                for line in r["report"][:30]:
                    sections.append(f"- {line}")
        
        return "\n".join(sections)

    def interactive(self) -> None:
        """Интерактивный режим — REPL."""
        print("\n🤖 Mansoni Interactive Mode. Введите задачу или 'exit' для выхода.\n")
        while True:
            try:
                task = input("mansoni> ").strip()
                if task.lower() in ("exit", "quit", "q"):
                    break
                if not task:
                    continue
                self.run(task)
            except KeyboardInterrupt:
                print("\n")
                break
            except Exception as e:
                logger.error(f"Ошибка: {e}", exc_info=True)


def main():
    parser = argparse.ArgumentParser(description="Mansoni — мультиагентный оркестратор")
    parser.add_argument("task", nargs="?", help="Задача текстом")
    parser.add_argument("--task-file", help="Файл с описанием задачи")
    parser.add_argument("--interactive", "-i", action="store_true", help="Интерактивный режим")
    parser.add_argument("--provider", default="auto", choices=["auto", "anthropic", "openai", "heuristic"])
    parser.add_argument("--project-root", default=str(_project_root))
    args = parser.parse_args()

    orchestrator = MansoniOrchestrator(args.project_root, args.provider)

    if args.interactive:
        orchestrator.interactive()
    elif args.task_file:
        task = Path(args.task_file).read_text(encoding="utf-8")
        orchestrator.run(task)
    elif args.task:
        orchestrator.run(args.task)
    else:
        parser.print_help()


if __name__ == "__main__":
    main()
'''

with open('ai_engine/bridge/mansoni_runner.py', 'w', encoding='utf-8') as f:
    f.write(RUNNER_CODE.strip())
print('✓ mansoni_runner.py записан')

In [ ]:
# ── __main__.py — чтобы работало python -m ai_engine.bridge ─────

MAIN_CODE = '''from ai_engine.bridge.mansoni_runner import main
main()
'''

with open('ai_engine/bridge/__main__.py', 'w', encoding='utf-8') as f:
    f.write(MAIN_CODE)
print('✓ __main__.py записан')

In [ ]:
# ── VS Code task ─────────────────────────────────────────────────
# Создаём tasks.json для запуска из VS Code

import json

vscode_dir = Path('.vscode')
vscode_dir.mkdir(exist_ok=True)

tasks_file = vscode_dir / 'tasks.json'

# Прочитаем существующий tasks.json если есть
existing_tasks = {"version": "2.0.0", "tasks": []}
if tasks_file.exists():
    try:
        existing_tasks = json.loads(tasks_file.read_text(encoding='utf-8'))
    except:
        pass

# Добавляем задачи Mansoni
mansoni_tasks = [
    {
        "label": "Mansoni: Interactive",
        "type": "shell",
        "command": "python",
        "args": ["-m", "ai_engine.bridge", "--interactive"],
        "isBackground": False,
        "problemMatcher": []
    },
    {
        "label": "Mansoni: Full Audit",
        "type": "shell",
        "command": "python",
        "args": ["-m", "ai_engine.bridge", "Полный аудит проекта: TypeScript, безопасность, архитектура, производительность"],
        "isBackground": False,
        "problemMatcher": []
    },
    {
        "label": "Mansoni: Run Tests",
        "type": "shell",
        "command": "python",
        "args": ["-m", "ai_engine.bridge", "Запусти все тесты, проверь покрытие, напиши недостающие тесты"],
        "isBackground": False,
        "problemMatcher": []
    },
    {
        "label": "Mansoni: Generate Docs",
        "type": "shell",
        "command": "python",
        "args": ["-m", "ai_engine.bridge", "Сгенерируй полную документацию проекта: архитектура, API, схема БД, модули"],
        "isBackground": False,
        "problemMatcher": []
    },
]

# Убираем дубли
existing_labels = {t.get('label') for t in existing_tasks.get('tasks', [])}
for mt in mansoni_tasks:
    if mt['label'] not in existing_labels:
        existing_tasks.setdefault('tasks', []).append(mt)

tasks_file.write_text(json.dumps(existing_tasks, indent=2, ensure_ascii=False), encoding='utf-8')
print('✓ .vscode/tasks.json обновлён')
print(f'  Mansoni задачи: {[t["label"] for t in mansoni_tasks]}')

In [ ]:
# ── Тест: проверяем что всё работает ─────────────────────────────

print('='*60)
print('  ТЕСТ: Проверка компонентов Mansoni')
print('='*60)

# 1. ProjectTools
from ai_engine.bridge.tool_implementations import ProjectTools
tools = ProjectTools(str(PROJECT_ROOT))
print(f'\n✓ ProjectTools инициализирован: {tools.root}')

# 2. Grep тест
result = tools.grep('export function ChatConversation', include='*.tsx')
print(f'✓ grep работает: нашёл {len(result.splitlines())} совпадений')

# 3. File read
content = tools.read_file('package.json', 1, 5)
print(f'✓ read_file работает: {len(content)} символов')

# 4. Line count
lines = tools.file_line_count('src/components/chat/ChatConversation.tsx')
print(f'✓ file_line_count: ChatConversation.tsx = {lines} строк')

# 5. tsc check
tsc = tools.tsc_check()
has_errors = 'error TS' in tsc
print(f'✓ tsc_check: {"ОШИБКИ" if has_errors else "0 ошибок"}')

# 6. Memory
mem = tools.read_memory('repo')
print(f'✓ read_memory: {len(mem)} символов загружено')

# 7. LLM provider
from ai_engine.bridge.llm_provider import create_llm_callable
llm = create_llm_callable(provider='auto')
print(f'✓ LLM provider: {"API" if "heuristic" not in str(llm) else "heuristic (нужен API ключ)"}')

# 8. Agent factory
from ai_engine.bridge.agent_factory import create_agent, AGENT_PROMPTS
print(f'✓ Agent factory: {len(AGENT_PROMPTS)} ролей доступно')
for role in AGENT_PROMPTS:
    print(f'    - {role}')

# 9. MansoniOrchestrator
from ai_engine.bridge.mansoni_runner import MansoniOrchestrator
orchestrator = MansoniOrchestrator(str(PROJECT_ROOT), llm_provider='heuristic')
classification = orchestrator.classify_task('Проведи полный аудит безопасности')
print(f'\n✓ MansoniOrchestrator: задача классифицирована как "{classification["type"]}"')
print(f'  Пайплайн: {" → ".join(classification["pipeline"])}')

print('\n' + '='*60)
print('  ВСЕ КОМПОНЕНТЫ РАБОТАЮТ ✓')
print('='*60)

## Как запускать

### Из терминала:
```bash
# Установить API ключ
set ANTHROPIC_API_KEY=sk-ant-...

# Интерактивный режим
python -m ai_engine.bridge --interactive

# Одиночная задача
python -m ai_engine.bridge "Проведи аудит безопасности Edge Functions"

# Полный аудит
python -m ai_engine.bridge "Полный аудит проекта"
```

### Из VS Code:
Ctrl+Shift+P → Tasks: Run Task → Mansoni: Interactive

### Из этого ноутбука:
```python
orchestrator = MansoniOrchestrator(PROJECT_ROOT)
report = orchestrator.run("Проведи аудит мессенджера")
```